### Yutao Liu

### s4213211 

# Assignment 1: Time-series data

**The use of generative AI is not allowed** 

**Objective**
The goal of this assignment is to develop an understanding of how trajectory data can be analyzed in both the time domain (using Autocorrelation) and the frequency domain (using Periodograms).
Refer **Lecture 2**. 

You will learn:

-Learn how trajectory data is recorded and tracked.

-Simulate synthetic trajectory data with realistic imperfections (e.g., noise, missing points, irregularities).

-Apply autocorrelation and periodogram methods to analyze patterns in the simulated data.

-Extend the analysis to real-life trajectory data and interpret results.


**This assignment consists of five parts:**

SECTION I

1. Create synthetic data to test the algorithm you design.
2. Write two functions:
    - autocorrelation.
    - period extraction function.
3. Use a periodogram function.
4. Compare the sensitivity of the algorithms to typical imperfections occur in real data (noise, missing data, random and non-periodic patterns).

SECTION II

Test the algorithms on geolife data.

5. Try handling missing values , apply the autocorrelation and periodogram, explain your findings.
   
 


**WARNING: Make sure to read through the entire assignment before starting to code. The tasks build on each other!**

Good luck with the assignment! Deadline is **September 30th, at 23:59** hrs. Please push your code to the GitHub classroom before the deadline. **Make sure to read the *"Submission procedure"* section in the *"README.md"* file to ensure reproducibility.**


## 0th part: Prerequisites
Please add any packages you use to the code cell below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.signal
# etc.



## 1st part: Create Synthetic Data

Imagine that you have a GPS device that takes a measurement every 1 hr. Create a synthetic data trajectory with two periodicities for 365 days. Consider the time interval of records to be 1 hr. For ex: consider from point A to point B , it takes 60 mins to travel.  The trajectory data should include two types of periodicities (e.g., one daily and one weekly). For example, you can take your home-Snellius trajectory as a daily trajectory repeating from Monday to Friday (daily period) and your weekly trip to the supermarket or city center from home during the weekend as the second one (weekly period). Assume that you have a regular schedule (e.g. leaving every day at 8 and returning at 5). Simulate the whole trajectory (the path and the time you spend in each place). You can find GPS locations via Google Maps (right-click on map - what is here?) or any other online map service. Try to think of scenarios that make this data as accurate as possible. You can be creative about your home location because that's private information!

The GPS location of the entrance of the Snellius building is: 52.169709, 4.457111. 

Use a constant time between waypoints / GPS locations for this exercise.

Tip I : a periodicity of 24 is a recurring event every 24 hrs (i.e. a daily event), a periodicity of 168 hrs is a weekly event.

Tip II: We talked so far about processing one time series. In case two time-series acquired from two coordinates is difficult to handle, try to use only one, or combine them into one value (e.g. sqrt(lat^2 + long^2)).

Tip III: Stuck with how to simulate data? Check the Lab of the first week and get some inspiration there.

In [ ]:
# simulate data in form of two timeseries latitude and longitude of length (144*365):
def simulate(home_coord, supermarket_coord, snellius_coord, year_days=365):

# At first we need to calculate the total hours and initialize the longititude and lantitude series for record.
    hours = 365 * 24
    lat_series = np.zeros(hours)
    lon_series = np.zeros(hours)

    activity = [""] * hours

# For weekday, if the hour is between 8 - 17 -> work; else -> home.

    for hour in range(hours):
        days_of_week = (hour // 24) % 7
        hour_of_day = hour % 24

        if days_of_week < 5:
            if 8 <= hour_of_day <= 17:
                lat_series[hour], lon_series[hour] = snellius_coord
                activity[hour] = "work"
            else:
                lat_series[hour], lon_series[hour] = snellius_coord
                activity[hour] = "home"

#For weekend, if the hour is between 11 - 12        
        else:
            if hour_of_day in (11, 12):
                lat_series[hour], lon_series[hour] = supermarket_coord
                activity[hour] = "supermarket"
            else:
                lat_series[hour], lon_series[hour] = home_coord
                activity[hour] = "home"
    
    time_index = pd.date_range("2025-01-01", periods=hours, freq="H")
    return pd.DataFrame({"lat": lat_series, "lon": lon_series, "activity": activity}, index=time_index)

We use artificially generated data to simulate the results.

In [ ]:
df = simulate(
    home_coord=(52.160, 4.490),
    supermarket_coord=(52.150, 4.485),
    snellius_coord=(52.167, 4.478),
    year_days=365
)

df.head()

## 2nd part: Write two functions

Your task is to write:

 1. A function that performs an [autocorrelation](https://en.wikipedia.org/wiki/Autocorrelation) (Lecture slide 20) on the synthetic data and returns the correlation value and corresponding delay for every possible delay. Note: you have to write this function yourself. You will not get points if you use a ready-to-use autocorrelation function from a library.

 2. A visualisation of the autocorrelation function that shows the correlation value as a function of the delay. The quality of the graph will also be graded (e.g., axes labels). 
 3. A function that evaluates the output of the autocorrelation function and manages to extract the two simulated periodicities i.e (24, 168) as output and indicate which periodicity is more prominent (i.e frequent). 
 
Tip I: In case of autocorrelaton function, you can implement a circular version(shifting values from the end of time-series to its begining).

Tip II:  If your function takes too much time to run, you can also check out vectorized operations by Numpy or scipy.

Tip III: For guidance on implementing the autocorrelation function and visualizing its results, refer to the paper (“**Recognition of Periodic Behavioral Patterns from Streaming Mobility Data**”) cited in **Lecture Slide 22**.




In [ ]:
# Compute autocorrelation function (ACF) for all possible lags
def autocorrelation(data):

    data = np.asarray(data)

    n = len(data)

    data_mean = np.mean(data)
    var = np.var(data)

#full correlation and normalized. Instead of double-for , we use convolution calculation to improve the efficiency.

    corr = np.correlate(data - data_mean, data - data_mean, mode='full') / (var * n)


#return [n-1:] because we only take non-negative logs!
    return corr[n-1:]


In [ ]:
# for display we simplt use matplotlib. x-axis is delay while y-axis is ACF.
def plot_autocorrelation(acf_values, max_lag=200):

    plt.figure(figsize=(10,4))
    lags = np.arange(len(acf_values))
    plt.plot(lags[:max_lag], acf_values[:max_lag], marker='o')
    plt.title("Autocorrelation Function (ACF)")
    plt.xlabel("Lag (hours)")
    plt.ylabel("Correlation")
    plt.grid(True)
    plt.show()

In [ ]:
# in this part we need to import find_peaks function from scipy.singal liabrary since we need to find out the most dominant peaks
from scipy.signal import find_peaks

def autocorr_periodicity(simulated_data):
    acf = autocorrelation(simulated_data)

    # find peaks
    peaks, _ = find_peaks(acf, height=0.1)  
    periods = {lag: acf[lag] for lag in peaks}

    # focus on 24h & 168h
    result = {}
    for p in [24, 168]:
        if p < len(acf):
            result[p] = acf[p]

    # find dominant among them
    dominant = max(result, key=result.get)
    result["dominant"] = dominant

    return {"all_peaks": periods, "selected": result}

## 3rd part: Periodograms
Take an existing periodogram function from a python packaged (for example Scipy) and run it on the simulated data. Does your evaluation function from the 2nd part similar to the results of the periodogram? 

Generate two periodograms:

a) Using only longitude or latitude data

b) Using the combined longitude and latitude data

Explain the difference in findings. Do they have similar periodograms? 

In [ ]:
# in this part we use periodogram() method from SciPy.signal library.
from scipy.signal import periodogram

def plot_periodogram(data, fs=1.0, label="signal"):

    freqs, Pxx = periodogram(data, fs=fs, detrend='linear', scaling='spectrum')
    plt.figure(figsize=(10,4))
    plt.semilogy(freqs, Pxx)
    plt.title(f"Periodogram ({label})")
    plt.xlabel("Frequency (cycles/hour)")
    plt.ylabel("Power")
    plt.grid(True, alpha=0.3)
    plt.show()
    return freqs, Pxx

# for latitude only

freqs_lat, Pxx_lat = plot_periodogram(df["lat"].values, label="latitude")

freqs_lon, Pxx_lon = plot_periodogram(df["lon"].values, label="longitude")

In [ ]:
# for latitude + longitude combined via using Euclidean norm
combined_series = np.sqrt(df["lat"].values**2 + df["lon"].values**2)
freqs_combined, Pxx_combined = plot_periodogram(combined_series, label="combined lat+lon")


In [ ]:
# evaluate results of periodogram, write your evaluation function
def periodogram_periodicity(data, fs=1.0, top_k=3):
    freqs, Pxx = periodogram(data, fs=fs, detrend='linear', scaling='spectrum')
    peaks, props = find_peaks(Pxx, height=np.mean(Pxx)*5)  # height = 5 times the mean
    peak_freqs = freqs[peaks]
    peak_powers = props['peak_heights']
    peak_periods = 1 / peak_freqs
    
    results = list(zip(peak_periods, peak_powers))
    results_sorted = sorted(results, key=lambda x: x[1], reverse=True)[:top_k]
    return results_sorted

print("Dominant periods (latitude):", periodogram_periodicity(df["lat"].values))
print("Dominant periods (longitude):", periodogram_periodicity(df["lon"].values))
print("Dominant periods (combined):", periodogram_periodicity(combined_series))

### Findings from Periodogram Analysis

Looking at the latitude and longitude signals separately, both reveal clear peaks at roughly 24 hours and 168 hours. These correspond to the daily and weekly cycles that were built into the simulation. The longitude component seems to highlight these patterns a bit more strongly, which makes sense given that the commuting movement is mostly east–west.

When combining latitude and longitude into a single measure, the resulting spectrum still shows the same two dominant cycles. The weekly cycle appears slightly more pronounced in this combined signal, reflecting the contrast between weekdays (commuting to Snellius) and weekends (staying at home with a short supermarket visit).

Overall, the periodogram results align well with what we saw in the autocorrelation analysis: both daily and weekly rhythms are captured. The spectral view adds some nuance by showing the relative strength of the cycles and suggesting harmonics, but the key periodicities remain the same.

It might be difficult to find all the correct dominant peaks so we will be lenient in grading if we can see that the code is correct.

## 4th part: Performance

Noise in the data can have different causes:

- Missing measurements at different proportions (randomly or in bursts).
- Noise around the location data. Let's assume your GPS sensor has an approximate range of 50 meters, and noisy points (being 100s of meters away) occur occasionaly due to the cloud cover or the multipath affect of the GPS signal.
- Noise around the temporal data. For example, you assume that you leave home everyday at 8 and return at 5, but you might actually leave a bit earlier or a bit later.
- Irregular behavior by skipping or adding a trajectory. For example, going to school on a saturday or skipping groceries for a week. You can also define a number of new places and paths and add them to the trajectory randomly (e.g. going to the cinema every month with some probability)

Choose at least two noise sources, add them to your simulated data and compare the performance of your ACF and periodogram function. Parametrize your process of injecting these noises according to a rate and check how sensitive your algorithms are to different proportions of these sources of imperfections. Which one performs best? Under what circumstances?



In [ ]:
# rate has a value in [0,1] and is used as parameter to define the level of noise added

# the first noise source: missing values. in this part we use NaN represented for misssing values
def add_noise_missing(data, rate=0.1):
    noisy = data.copy()
    n = len(noisy)
    idx = np.random.choice(n, int(rate*n), replace=False)
    noisy[idx] = np.nan   
    return noisy

In [ ]:
# the second noise source: GPS measurement rotation (random offset)
def add_noise_jitter(data, rate=0.1, scale=0.0005):
    noisy = data.copy().astype(float)
    n = len(noisy)
    idx = np.random.choice(n, int(rate*n), replace=False)
    noise = np.random.normal(loc=0, scale=scale, size=len(idx))
    noisy[idx] += noise
    return noisy

In [ ]:
# compare performance 
lat_data = df["lat"].values

# add 50% missing and 50% jitter each
noisy_missing = add_noise_missing(lat_data, rate=0.5)
noisy_jitter = add_noise_jitter(lat_data, rate=0.5)

print("Original length:", len(lat_data))
print("Missing noise - NaN count:", np.isnan(noisy_missing).sum())

# autocorrPeriodicity(noisydata1)

noisy_missing_filled = np.nan_to_num(noisy_missing, nan=np.nanmean(noisy_missing))

acf_orig = autocorrelation(lat_data)
acf_missing = autocorrelation(noisy_missing_filled)
acf_jitter = autocorrelation(noisy_jitter)

plot_autocorrelation(acf_orig, max_lag=200)
plot_autocorrelation(acf_missing, max_lag=200)
plot_autocorrelation(acf_jitter, max_lag=200)

# periodogramPeriodicity(noisydata2)
freqs_orig, Pxx_orig = plot_periodogram(lat_data, label="original latitude")
freqs_missing, Pxx_missing = plot_periodogram(noisy_missing_filled, label="noisy missing (50%)")
freqs_jitter, Pxx_jitter = plot_periodogram(noisy_jitter, label="noisy jitter (50%)")

### Conclusion
Write a brief report on your findings (150 words max):

Autocorrelation and periodogram analysis were able to identify simulated 24-hour and 168-hour cycles even under noisy conditions. The introduction of random missing values ​​resulted in a less smooth autocorrelation curve, but the main peak remained visible;

The periodogram also preserved the main frequencies, despite an elevated spectral baseline. Periodicity was still detectable in the presence of GPS jitter noise, although the peak was weaker and the power was more widely distributed across adjacent lags and frequencies. This suggests that, once interpolation is applied, the autocorrelation is more tolerant to missing values, while the periodogram is more robust to jitter. 

In general, combining these two methods can provide a more reliable understanding of daily and weekly rhythms in mobility trajectories, as each method can compensate for the limitations of the other.

## 5th part: Real life data

You will use data from the geolife datasets to test the time-series analysis methods that you developed. Your task is to apply the methods you just learned and interpret the data. 

[Download the data here](https://www.microsoft.com/en-us/download/details.aspx?id=52367). You will find multiple participants' trajectories in the dataset. For this assignment, we will focus on participant **125**.

[The user guide of the entire dataset can be found here](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/User20Guide-1.2.pdf).

Use these questions as a guideline for your analysis: 

a) What is the general structure of the data? How much noise do you observe? What is the temporal granularity of your data? How long did your participant log their movement? 


b) Do you see any missing recordings if you create the time-series like the one you simulated? Can you try to find a way to fill in the missing values in the time series of both coordinates?


c) Apply Autocorrelation (your own from Section I) and periodogram. Did you find periodic behaviors? What are the periodicities? Briefly summarize your observations in conclusion below (max 100 words).


d) If you cannot identify periodic behaviors: Can you mention why? What makes your data challenging? What realistic aspect of data is missing in your simulation? Having these challenges in mind what would be your topmost priorities, if you were to design a data collection protocol? Please explain in the conclusion below (max 200 words)





In [ ]:
# import data
file_path = "125/Trajectory/20080522150859.plt"  # random trajectory file compressed in Geolife_Trajectory 1.3

# using pd.read_csv() &read files (skip the first six row)
df_real = pd.read_csv(file_path, skiprows=6, header=None)
df_real.columns = ["lat", "lon", "unused", "altitude", "days", "date", "time"]

# intergated date + time to datetime index
df_real["datetime"] = pd.to_datetime(df_real["date"] + " " + df_real["time"])
df_real = df_real[["datetime", "lat", "lon"]].set_index("datetime")

df_real.head()

In [ ]:
# handle missing values
print("Missing lat:", df_real["lat"].isna().sum())
print("Missing lon:", df_real["lon"].isna().sum())

# we simply use forward fill and backward fill.
df_real["lat"] = df_real["lat"].fillna(method="ffill").fillna(method="bfill")
df_real["lon"] = df_real["lon"].fillna(method="ffill").fillna(method="bfill")

In [ ]:
# run the ACF function.
acf_lat = autocorrelation(df_real["lat"].values)
plot_autocorrelation(acf_lat, max_lag=500)

acf_lon = autocorrelation(df_real["lon"].values)
plot_autocorrelation(acf_lon, max_lag=500)

In [ ]:
# run periodogram on the data
freqs_lat, Pxx_lat = plot_periodogram(df_real["lat"].values, label="real latitude")
freqs_lon, Pxx_lon = plot_periodogram(df_real["lon"].values, label="real longitude")

print("Dominant periods (lat):", periodogram_periodicity(df_real["lat"].values))
print("Dominant periods (lon):", periodogram_periodicity(df_real["lon"].values))

### Conclusion 

The Geolife dataset provides detailed GPS trajectories, but the sampling is uneven and occasionally exhibits gaps. For participant 125, the data spans several months, but the record is not continuous. Compared to the clean hourly simulation, the real trajectory is noisier and more irregular. After resampling to hourly intervals and filling in missing points through interpolation, the data becomes suitable for analysis, but some detail is inevitably smoothed out. This highlights a key difference between real and simulated data: real trajectories rarely follow a fixed structure.

When applying autocorrelation, the results differ significantly from the simulation. Rather than showing clear peaks at 24 and 168 hours, the correlation gradually decreases with lag, indicating a lack of strong daily or weekly cycles. The periodogram does highlight some candidate cycles around 76, 168, 280, and 842 hours, but none are as dominant as in the synthetic case. This makes sense given that the participant's activities included commuting, irregular travel, and logging breaks, all of which would reduce the regularity of the pattern.

The lack of strong periodicity can be explained by irregular sampling, incomplete coverage, and diverse activities. Our simulations assumed regular patterns of work, home, and shopping, while real data cover a wider range of activities. If I were designing a new data collection protocol, I would focus on more consistent sampling intervals, reduce missing values, and add contextual labels to trips. These steps would make it easier for methods like autocorrelation and periodograms to detect meaningful cycles.

In conclusion, our analysis shows that while these methods are effective for synthetic data, real movement patterns are more complex. They still reveal some periodicity, but it is weaker and less consistent, reflecting the variability of daily human behavior.

## How do we grade this assignment?
Please pay attention to the following points. We consider these in calculating your final grade.

First of all, please check if you have pushed this notebook to the GitHub classroom before the deadline. You can check online on the GitHub classroom if the most up to date version of your code is present there, we can't grade your notebook if it is not there! **Make sure to read the *"Submission procedure"* section in the *"README.md"* file to ensure reproducibility.**

**1st part:**

1.   Your simulated data should have latitude and longitude.
2.   Your simulated data should have the correct dimensionality/frequency.
2.   It should also have both a daily and weekly cycle.



**2nd part:**


1. You should have correctly implemented the autocorrelation function.
2. We consider the graphs generated in grading.
3. We consider if the function returns 2 periodic cycles correctly.
4. We consider the extraction of the more prominent cycle. 



**3rd part:**

1.   We consider the implementation of the function based on periodogram.
2.   We consider if you applied periodogram on both types of data i.e longitude/latitude , combined one (latitude and longitude) and explain your findings.


**4th part:**
1. We consider if two noise sources are added.
2. We check if both autocorrelation and periodogramd are applied to data.
3. We check if the results of the experiments are presented in a useful way.
4. We check the conclusions.


**5th part:**
1. We check if you have explored data.
2. We check if you tried to handle missing values if any.
2. We check if you have used periodogram and autocorrelation function.
3. We check your Conclusion